In [ ]:
!pip install gsw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.7 MB/s eta 0:00:00


In [ ]:
import os, glob, re, warnings
import numpy as np
import pandas as pd
import xarray as xr
import gsw

# ---------------- USER CONFIG ----------------
DIR = "/content/Floats"
OUT_CSV = os.path.join(DIR, "ancp_johnson_aligned_verbose.csv")  # set None to skip writing

ZMAX = 200.0
DZ = 2.0
SURF_MAX = 30.0
PRINT_QA = True

NO3_CANDIDATES = ["NTAW", "NITRATE", "NO3"]
warnings.filterwarnings("ignore", message="Mean of empty slice")
# --------------------------------------------


# ---------------- small helpers ----------------
def _wmo_from_ds(ds, fname: str) -> str:
    """
    Extract the WMO (World Meteorological Organization) platform number
    for an Argo float dataset.

    The function first attempts to read the platform identifier from the
    dataset variable 'PLATFORM_NUMBER'. If present, it converts the value
    to a clean string (removing whitespace and handling array formatting).

    If 'PLATFORM_NUMBER' is missing or cannot be parsed, the function falls
    back to extracting a numeric identifier (>=6 digits) from the input
    filename. If no such pattern is found, the basename of the file is
    returned as a last resort.
    """

    if "PLATFORM_NUMBER" in ds.variables:
        try:
            raw = ds["PLATFORM_NUMBER"].values
            wmo = "".join(np.atleast_1d(raw)[0].astype(str)).strip()
            return re.sub(r"\s+", "", wmo)
        except Exception:
            pass
    m = re.search(r"(\d{6,})", os.path.basename(fname))
    return m.group(1) if m else os.path.basename(fname)


def _resolve_var(ds, base_names):
    """Return (DataArray, base_name_used) preferring *_ADJUSTED."""
    if isinstance(base_names, str):
        base_names = [base_names]
    for b in base_names:
        an = b + "_ADJUSTED"
        if an in ds.variables:
            return ds[an], b
    for b in base_names:
        if b in ds.variables:
            return ds[b], b
    return None, None


def _apply_qc(ds, da, base: str):
    """
    Apply QC if an exactly-dimension-matching QC variable exists.
    Keeps QC flags 1,2,8 (plus implicitly others if you add).
    """
    for suf in ("_ADJUSTED_QC", "_QC"):
        qn = base + suf
        if qn in ds.variables and tuple(ds[qn].dims) == tuple(da.dims):
            qc = ds[qn].astype("U1")
            return da.where(qc.isin(list("128")))
    return da


def _units_kind(units_attr: str) -> str:
    """
    Classify nitrate concentration units into a normalized unit category.

    The function standardizes a units attribute string by removing
    non-ASCII characters (to handle corrupted micro symbols), normalizing
    common textual variants, and then mapping the units to a small set of
    expected nitrate unit types.

    It is designed for robustness against formatting inconsistencies
    commonly found in Argo and biogeochemical datasets.
    """

    if not units_attr:
        return "unknown"

    # Handle corrupted micro sign like '\udcc2\udcb5' by stripping non-ascii.
    u = str(units_attr).encode("ascii", "ignore").decode("ascii").strip().lower()

    # Normalize common variants
    u = u.replace("µ", "u")  # harmless if µ exists, usually removed by ascii ignore

    # Be robust: your cleaned string might become "mol kg-1" (lost the 'u'),
    # but we still want to treat any "*mol/kg" nitrate as micro-moles per kg in Argo context.
    if ("kg-1" in u) or ("kg^-1" in u) or ("/kg" in u):
        if "mol" in u:
            return "umol_kg"

    # mmol per m^3 (also accept mmol/m3 and mmol m-3)
    if ("mmol/m3" in u) or ("mmol m-3" in u) or ("mmol m^-3" in u):
        return "mmol_m3"

    # Some datasets might store umol/L, which is numerically mmol/m^3.
    if ("umol/l" in u) or ("umol l-1" in u) or ("umol l^-1" in u):
        return "mmol_m3"

    return "unknown"



def _decode_time(ds):
    """
    Decode CF-compliant time coordinates into a pandas-compatible index.

    The function applies xarray's CF decoding to interpret encoded time
    variables (e.g., "days since 1950-01-01") and converts the TIME
    coordinate into a pandas DatetimeIndex when possible. If direct
    conversion fails, it falls back to pandas datetime parsing.
    """

    dsd = xr.decode_cf(ds)
    t = dsd["TIME"]
    try:
        return t.to_index()
    except Exception:
        return pd.to_datetime(t.values)


def _lat_at(ds, i: int) -> float:
    """
    Retrieve latitude for a specified profile index.

    The function handles both scalar and vector latitude variables.
    If latitude varies by profile, the value at index `i` is returned.
    If the requested index exceeds bounds, the last available value
    is used.
    """

    if "LATITUDE" not in ds:
        return np.nan
    da = ds["LATITUDE"]
    if da.ndim == 0:
        return float(da.values)
    d0 = da.dims[0]
    return float(da.isel({d0: min(i, da.sizes[d0] - 1)}).values)


def _lon_at(ds, i: int) -> float:
    """
    Retrieve longitude for a specified profile index.

    The function supports both scalar and vector longitude variables.
    If longitude varies by profile, the value at index `i` is returned.
    If the requested index exceeds bounds, the last available value
    is used.
    """

    if "LONGITUDE" not in ds:
        return np.nan
    da = ds["LONGITUDE"]
    if da.ndim == 0:
        return float(da.values)
    d0 = da.dims[0]
    return float(da.isel({d0: min(i, da.sizes[d0] - 1)}).values)


def _depth_from_p(pres_1d, lat: float):
    """
    Convert pressure to depth using TEOS-10 seawater equations.

    The function computes depth (meters, positive downward) from
    pressure using the Gibbs SeaWater (GSW) library. If the conversion
    fails, NaNs are returned.
    """

    p = np.asarray(pres_1d, float)
    try:
        return -gsw.z_from_p(p, float(lat))  # meters, positive down
    except Exception:
        return np.full_like(p, np.nan, float)


def _to_mmol_m3_from_umol_kg(no3_umolkg, pres, psal, temp, lat):
    """
    Convert nitrate from µmol/kg to mmol/m³.

    The conversion uses TEOS-10 seawater density derived from
    Absolute Salinity (SA) and Conservative Temperature (CT),
    computed from practical salinity, in-situ temperature,
    pressure, and latitude.
    """

    SA = gsw.SA_from_SP(psal, pres, 0.0, lat)
    CT = gsw.CT_from_t(SA, temp, pres)
    rho = gsw.rho(SA, CT, pres)  # kg m^-3
    return (no3_umolkg * rho) / 1000.0  # (umol/kg * kg/m^3)/1000 = mmol/m^3


def _ensure_mmol_m3(NO3, P, S, T, lat, i, units_kind, time_dim, warn_prefix=""):
    """Return nitrate in mmol/m^3 for profile i along time_dim."""
    no3_i = np.asarray(NO3.isel({time_dim: i}).values, float)

    if units_kind == "mmol_m3":
        return no3_i

    p_i = np.asarray(P.isel({time_dim: i}).values, float) if P is not None else None
    s_i = np.asarray(S.isel({time_dim: i}).values, float) if S is not None else None
    t_i = np.asarray(T.isel({time_dim: i}).values, float) if T is not None else None

    if units_kind == "umol_kg":
        if (p_i is not None) and (s_i is not None) and (t_i is not None) and np.isfinite(lat):
            return _to_mmol_m3_from_umol_kg(no3_i, p_i, s_i, t_i, lat)
        if warn_prefix:
            print(f"[WARN]{warn_prefix} umol/kg but missing P/S/T/lat; treating as mmol/m^3.")
        return no3_i

    if warn_prefix:
        print(f"[WARN]{warn_prefix} units unknown; treating as mmol/m^3.")
    return no3_i


def _interp_to_grid(z, v, zgrid, min_pts=3):
    """
    Interpolate a profile variable onto a target depth grid.

    The function filters out non-finite depth and variable values,
    requires a minimum number of valid points, sorts the profile by
    depth, and performs 1D linear interpolation onto the provided
    grid. Interpolation is only applied within the observed depth
    range, rest are set to NaN.
    """

    z = np.asarray(z, float)
    v = np.asarray(v, float)
    m = np.isfinite(z) & np.isfinite(v)
    if m.sum() < min_pts:
        return np.full_like(zgrid, np.nan, float)

    zz = z[m]
    vv = v[m]
    o = np.argsort(zz)
    zz = zz[o]
    vv = vv[o]

    out = np.full_like(zgrid, np.nan, float)
    mask = (zgrid >= zz.min()) & (zgrid <= zz.max())
    out[mask] = np.interp(zgrid[mask], zz, vv)
    return out


def _surface_mean_0_Z_from_grid(vgrid, zgrid, zsurf):
    """
    Compute the mean concentration within the surface layer.

    The function integrates a gridded vertical profile from the surface
    (0m) down to a specified depth ('zsurf') using trapezoidal
    integration, then divides by the depth span to obtain the layer
    mean. Only finite values within the specified depth range are used.

    A minimum of two valid depth points is required to compute the mean.
    """

    z = np.asarray(zgrid, float)
    v = np.asarray(vgrid, float)
    m = np.isfinite(z) & np.isfinite(v) & (z >= 0) & (z <= zsurf)
    if m.sum() < 2:
        return np.nan

    zz = z[m]
    vv = v[m]
    o = np.argsort(zz)
    zz = zz[o]
    vv = vv[o]

    integ = np.trapezoid(vv, zz)
    span = zz[-1] - zz[0]
    return float(integ / span) if span > 0 else np.nan


def _concat_time(files):
    """Open and concat along TIME. Returns (ds_concat, opened_dsets)."""
    dsets = []
    for p in files:
        try:
            dsets.append(xr.open_dataset(p))
        except Exception:
            pass
    if not dsets:
        return None, []

    if len(dsets) == 1:
        return dsets[0], dsets

    try:
        ds = xr.concat(
            dsets, dim="TIME",
            data_vars="minimal", coords="minimal",
            compat="override", join="outer"
        )
        return ds, dsets
    except Exception:
        # fall back: concat sequentially
        ds = dsets[0]
        for ex in dsets[1:]:
            try:
                ds = xr.concat(
                    [ds, ex], dim="TIME",
                    data_vars="minimal", coords="minimal",
                    compat="override", join="outer"
                )
            except Exception:
                pass
        return ds, dsets
# ---------------------------------------------


def ancp_johnson_simple_verbose():
    """
    Simplified Johnson-style ANCP:
    - for each calendar year y:
        winter candidate = Aug-Nov of y (pick month with max surface mean NO3)
        summer candidate = Dec(y) + Jan-Mar(y+1) (pick month with min surface mean NO3)
    - compute monthly mean profiles on 0..200m (2m grid), integrate (W-S) to 200m
    - ANCP = 6.6 * integral in mmol N m^-2 -> mmol C m^-2, also mol C m^-2
    """
    rows = []
    zgrid = np.arange(0.0, ZMAX + DZ / 2, DZ)

    # group files by float id (WMO)
    groups = {}
    for p in sorted(glob.glob(os.path.join(DIR, "*.nc"))):
        try:
            with xr.open_dataset(p) as ds0:
                no3, _ = _resolve_var(ds0, NO3_CANDIDATES)
                if no3 is not None:
                    wmo = _wmo_from_ds(ds0, p)
                    groups.setdefault(wmo, []).append(p)
        except Exception:
            pass

    if not groups:
        print("[ANCP] No files with nitrate-like variable found.")
        return pd.DataFrame()

    for wmo, files in groups.items():
        if PRINT_QA:
            print("\n" + "=" * 72)
            print(f"WMO {wmo} - concatenating and QC")

        ds, opened = _concat_time(files)
        if ds is None:
            continue

        try:
            NO3, no3_base = _resolve_var(ds, NO3_CANDIDATES)
            P, p_base = _resolve_var(ds, ["PRES", "PRESSURE"])
            S, s_base = _resolve_var(ds, ["PSAL", "SP", "SAL", "SALINITY"])
            T, t_base = _resolve_var(ds, ["TEMP", "TEMP_CTD", "TEMPERATURE"])

            if (NO3 is None) or ("TIME" not in ds.variables and "TIME" not in ds.coords):
                continue

            # QC
            NO3 = _apply_qc(ds, NO3, no3_base).where(NO3 >= 0)
            if P is not None and p_base is not None:
                P = _apply_qc(ds, P, p_base)
            if S is not None and s_base is not None:
                S = _apply_qc(ds, S, s_base)
            if T is not None and t_base is not None:
                T = _apply_qc(ds, T, t_base)

            ukind = _units_kind(NO3.attrs.get("units", ""))
            tt = _decode_time(ds)
            n = len(tt)

            time_dim = NO3.dims[0]  # profile dimension

            if PRINT_QA:
                print(f"Profiles: {n} | units: {NO3.attrs.get('units','NA')} -> kind: {ukind}")

            # precompute per-profile surface mean and gridded profile
            prof = []
            for i in range(n):
                lat = _lat_at(ds, i)
                lon = _lon_at(ds, i)

                if P is None:
                    prof.append(dict(
                        time=tt[i], year=tt[i].year, month=tt[i].month,
                        lat=lat, lon=lon, surf_mean=np.nan, grid=None
                    ))
                    continue
                # Getting pressure from the data and converting it to depth using the helper function.
                p_i = np.asarray(P.isel({time_dim: i}).values, float)
                z_i = _depth_from_p(p_i, lat)

                v_mmolm3 = _ensure_mmol_m3(
                    NO3, P, S, T, lat, i, ukind, time_dim,
                    warn_prefix=f"[{wmo}] {tt[i].date()}"
                )
                # keep in mind that zgrid is defined at the top of this function. We create a grid and interpolate the target values onto the grid and compute the surface mean using the other functions above.
                grid = _interp_to_grid(z_i, v_mmolm3, zgrid)
                surf_mean = _surface_mean_0_Z_from_grid(grid, zgrid, SURF_MAX)

                # We append the data in the form of a dictionary so that we can convert it to a pandas ddataframe later. Note that the surface mean and the month is part of this data and is computed at this step itself.
                prof.append(dict(
                    time=tt[i], year=tt[i].year, month=tt[i].month,
                    lat=lat, lon=lon, surf_mean=surf_mean, grid=grid
                ))

            # Create the dataframe and sort the values with respect to time.
            df = pd.DataFrame(prof).sort_values("time")
            if df.empty:
                continue
            # Creating a list of years to iterate over. df["year"] would return a single column of all the years and .unique() would just return the unique years in our dataframe.
            years = sorted(df["year"].unique())
            if PRINT_QA:
                print(f"Years: {years}")

            # We loop over every year to filter the suitable data appropriately to compute ANCP
            for y in years:
                # Wcand and Scand are dataframes. Wcand contains the data of months from 8-11 of year y and Scand contains month 12 of year y and 1-3 of year y+1, i.e., the subsequent year as noted in the ANCP formula.
                Wcand = df[(df["year"] == y) & (df["month"].isin([8, 9, 10, 11]))].copy()
                Scand = df[((df["year"] == y) & (df["month"] == 12)) |
                           ((df["year"] == y + 1) & (df["month"].isin([1, 2, 3])))]
                if Wcand.empty or Scand.empty:
                    continue
                #idxmax returns the index of the DataFrame at which the maximum surf_mean occurs in and vice versa for idxmin()
                iw = Wcand["surf_mean"].idxmax()
                is_ = Scand["surf_mean"].idxmin()
                if pd.isna(iw) or pd.isna(is_):
                    continue
                # Using the index, we can obtain the other columns/features of that particular row of data and in this case, we obtain the month of the profile.
                mw = int(Wcand.loc[iw, "month"])
                ms = int(Scand.loc[is_, "month"])
                # Using the month obtained above, we obtain all the profiles of that month below.
                Wm = Wcand[Wcand["month"] == mw].copy()
                Sm = Scand[Scand["month"] == ms].copy()

                # We just stack all the grids in Wm and Sm. Think of it as an array of arrays.
                # if length of Wm is greater than zero, we make an array of the values in the grid of the first row of the dataframe and append it to an array. We continue making an array of values of the grids in the subsequent rows
                # until the last row and keep appending it to the array in order to have an array of arrays.
                Wgrids = np.stack([g for g in Wm["grid"].values if g is not None], axis=0) if len(Wm) else np.empty((0, zgrid.size))
                Sgrids = np.stack([g for g in Sm["grid"].values if g is not None], axis=0) if len(Sm) else np.empty((0, zgrid.size))

                if (Wgrids.size == 0) or (Sgrids.size == 0):
                    if PRINT_QA:
                        print("    skip: no gridded profiles for monthly mean")
                    continue
                # At each depth level of the grid, we take the mean of all non NaN values and assign it to that depth.
                Wmean = np.nanmean(Wgrids, axis=0)
                Smean = np.nanmean(Sgrids, axis=0)

                valid = np.isfinite(Wmean) & np.isfinite(Smean)
                if PRINT_QA and valid.any():
                    print(
                        f"  {wmo} {y}: W={mw} S={ms} nW={len(Wm)} nS={len(Sm)} | "
                        f"Overlap {zgrid[valid][0]:.1f}-{zgrid[valid][-1]:.1f} m (n={valid.sum()})"
                    )
                    nWc = np.sum(np.isfinite(Wgrids), axis=0)
                    nSc = np.sum(np.isfinite(Sgrids), axis=0)
                    print(
                        "    Contributors (median/min): "
                        f"W={np.nanmedian(nWc[valid]):.1f}/{np.nanmin(nWc[valid]):.0f}, "
                        f"S={np.nanmedian(nSc[valid]):.1f}/{np.nanmin(nSc[valid]):.0f}"
                    )

                diff = Wmean - Smean
                # We create a mask in such a way where we only accept finite values within 0-200m. ZMAX declared at the top of the code cell.
                m = np.isfinite(diff) & (zgrid >= 0) & (zgrid <= ZMAX)
                if m.sum() < 2:
                    if PRINT_QA:
                        print("    skip: insufficient overlap for integration")
                    continue
                # np.trapz just integrates the values of diff[m] over the interval of zgrid[m]
                Ninv = float(np.trapezoid(diff[m], zgrid[m]))  # mmol N m^-2
                ancp_mmolC = 6.6 * Ninv
                ancp_molC = ancp_mmolC / 1000.0

                if PRINT_QA:
                    print(f"    ANCP: {ancp_molC:.3f} mol C m^-2 ({ancp_mmolC:.1f} mmol C m^-2)")
                # We append our results in the form of dict so that we can convert it to a pandas DataFrame and subsequently a .csv file.
                rows.append(dict(
                    float_wmo=wmo,
                    season_calendar_year=y,
                    winter_month=mw,
                    summer_month=ms,
                    ancp_molC_m2=ancp_molC,
                    ancp_mmolC_m2=ancp_mmolC,
                    n_prof_winter=int(Wm.shape[0]),
                    n_prof_summer=int(Sm.shape[0]),
                ))

        finally:
            for d in opened:
                try:
                    d.close()
                except Exception:
                    pass

    out = (pd.DataFrame(rows)
           .sort_values(["float_wmo", "season_calendar_year"])
           .reset_index(drop=True))

    if out.empty:
        print("[ANCP] No rows.")
        return out

    if PRINT_QA:
        print("\nSummary table:")
        show = out[["float_wmo", "season_calendar_year", "winter_month", "summer_month",
                    "ancp_molC_m2", "n_prof_winter", "n_prof_summer"]]
        with pd.option_context("display.max_rows", 200, "display.width", 160):
            print(show.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

    if OUT_CSV:
        out.to_csv(OUT_CSV, index=False)
        print("Wrote:", OUT_CSV)

    return out


# ---- run ----
ancp_aligned_verbose = ancp_johnson_simple_verbose()



WMO 5904183 - concatenating and QC
Profiles: 233 | units: µmol kg-1 -> kind: umol_kg
Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
  5904183 2017: W=10 S=3 nW=3 nS=3 | Overlap 22.0-200.0 m (n=90)
    Contributors (median/min): W=3.0/1, S=3.0/2
    ANCP: 2.017 mol C m^-2 (2016.8 mmol C m^-2)

WMO 5904397 - concatenating and QC


/tmp/ipython-input-874710121.py:339: FutureWarning: The behavior of Series.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  is_ = Scand["surf_mean"].idxmin()
/tmp/ipython-input-874710121.py:338: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  iw = Wcand["surf_mean"].idxmax()
/tmp/ipython-input-874710121.py:339: FutureWarning: The behavior of Series.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  is_ = Scand["surf_mean"].idxmin()
/tmp/ipython-input-874710121.py:338: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  iw = Wcand["surf_mean"].idxmax()
/tmp/ipython-input-874710121.py:339: FutureWarning: The behavior of Series.idxmin with all

Profiles: 50 | units: µmol kg-1 -> kind: umol_kg
Years: [np.int64(2016), np.int64(2018), np.int64(2019)]
  5904397 2018: W=11 S=1 nW=3 nS=3 | Overlap 20.0-200.0 m (n=91)
    Contributors (median/min): W=3.0/1, S=3.0/3
    ANCP: 0.627 mol C m^-2 (627.3 mmol C m^-2)

WMO 5904467 - concatenating and QC
Profiles: 41 | units: µmol kg-1 -> kind: umol_kg
Years: [np.int64(2016), np.int64(2017)]
  5904467 2016: W=10 S=2 nW=3 nS=3 | Overlap 22.0-200.0 m (n=90)
    Contributors (median/min): W=3.0/2, S=3.0/3
    ANCP: 0.574 mol C m^-2 (574.1 mmol C m^-2)

WMO 5904472 - concatenating and QC
Profiles: 144 | units: µmol kg-1 -> kind: umol_kg
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
  5904472 2015: W=11 S=3 nW=3 nS=3 | Overlap 8.0-200.0 m (n=97)
    Contributors (median/min): W=3.0/1, S=3.0/1
    ANCP: 0.997 mol C m^-2 (996.6 mmol C m^-2)
  5904472 2016: W=11 S=1 nW=3 nS=3 | Overlap 8.0-200.0 m (n=97)
    Contributors (median/min): W=3.0/1, S=3.0/3
    A